# 小规模 QUBO Solver 验证

这本 notebook 使用仓库的 `qubo.v1` / `qubo-result.v1` 契约，对小规模 solver 做可重复验证。

验证顺序刻意分成四层：

1. 先验证输入 QUBO 的结构契约；
2. 用独立穷举建立 oracle，不依赖任何 solver；
3. 分别运行 Exact 与 QAOA solver，并验证结果契约；
4. 比较 sample、energy、status 和 optimality gap。

> QAOA 是近似算法，正确行为是返回 `feasible`，而不是宣称 `optimal`。

## 1. 环境与导入

请在 VS Code/Jupyter 中选择项目的 `.venv` kernel。下面的代码会自动向上寻找仓库根目录，因此从仓库根目录或 `notebooks/` 目录启动都可以。

In [1]:
import copy
import json
import sys
from pathlib import Path


PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "lib").is_dir() or not (PROJECT_ROOT / "problem").is_dir():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("Could not locate the QSolutionData repository root.")
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from lib.contracts import validate_qubo, validate_qubo_result
from lib.solvers.qubo import ExactQuboSolver, QaoaQuboSolver
from tests.oracles import enumerate_qubo, evaluate_qubo_energy, public_json_number

print("Project root:", PROJECT_ROOT)
print("Python:", sys.executable)

Project root: C:\Users\peter\Desktop\taiyi\QSolutionData
Python: c:\Users\peter\Documents\TaiyiQSolution\.venv\Scripts\python.exe


## 2. 定义一个 `qubo.v1`

这里构造三个二进制变量的最小化问题：每个变量有不同 reward，同时选择多个变量会产生 pair penalty。

仓库采用上三角稀疏项：`[i, j, coefficient]` 表示 $Q_{ij}x_ix_j$，offset 单独保存。

In [2]:
qubo = {
    "schema": "qubo.v1",
    "problem_id": "notebook-small-qubo",
    "sense": "minimize",
    "num_variables": 3,
    "variable_names": ["asset_alpha", "asset_beta", "asset_gamma"],
    "offset": 1.0,
    "terms": [
        [0, 0, -3.0],
        [1, 1, -2.0],
        [2, 2, -1.0],
        [0, 1, 4.0],
        [0, 2, 2.5],
        [1, 2, 2.0],
    ],
    "metadata": {
        "purpose": "small deterministic solver validation",
        "expected_unique_optimum": [1, 0, 0],
    },
}
qubo_snapshot = copy.deepcopy(qubo)

validate_qubo(qubo)
print(json.dumps(qubo, indent=2, ensure_ascii=False))

{
  "schema": "qubo.v1",
  "problem_id": "notebook-small-qubo",
  "sense": "minimize",
  "num_variables": 3,
  "variable_names": [
    "asset_alpha",
    "asset_beta",
    "asset_gamma"
  ],
  "offset": 1.0,
  "terms": [
    [
      0,
      0,
      -3.0
    ],
    [
      1,
      1,
      -2.0
    ],
    [
      2,
      2,
      -1.0
    ],
    [
      0,
      1,
      4.0
    ],
    [
      0,
      2,
      2.5
    ],
    [
      1,
      2,
      2.0
    ]
  ],
  "metadata": {
    "purpose": "small deterministic solver validation",
    "expected_unique_optimum": [
      1,
      0,
      0
    ]
  }
}


## 3. 独立穷举 oracle

这里调用 `tests.oracles` 中独立维护的 `Fraction` 穷举器。它不调用 solver 或 production energy helper；notebook 只负责调用、展示和断言。

In [3]:
truth_table = enumerate_qubo(qubo)
oracle_sample = truth_table[0]["sample"]
oracle_energy = truth_table[0]["energy_exact"]

for row in truth_table:
    print(row["sample"], "->", float(row["energy_exact"]))

assert oracle_sample == [1, 0, 0]
assert oracle_energy == -2
print("Oracle optimum:", oracle_sample, "energy =", float(oracle_energy))

[1, 0, 0] -> -2.0
[0, 1, 0] -> -1.0
[1, 0, 1] -> -0.5
[0, 0, 1] -> 0.0
[0, 1, 1] -> 0.0
[1, 1, 0] -> 0.0
[0, 0, 0] -> 1.0
[1, 1, 1] -> 3.5
Oracle optimum: [1, 0, 0] energy = -2.0


## 4. Exact solver

`ExactQuboSolver` 会枚举全部 $2^n$ 个 assignment。它适合作为小问题正确性 oracle，不适合大规模生产问题。

In [4]:
exact_solver = ExactQuboSolver()
exact_config = {"max_variables": 10}
exact_config_snapshot = copy.deepcopy(exact_config)
exact_result = exact_solver.solve(
    qubo,
    config=exact_config,
)
validate_qubo_result(qubo, exact_result)

assert exact_result["status"] == "optimal"
assert exact_result["termination_reason"] == "search_exhausted"
exact_energy_exact = evaluate_qubo_energy(
    qubo,
    exact_result["best_sample"],
)
assert exact_result["best_sample"] == oracle_sample
assert exact_energy_exact == oracle_energy
assert exact_result["best_energy"] == public_json_number(exact_energy_exact)
assert exact_result["metrics"]["search_space_exhausted"] is True
assert exact_result["metrics"]["candidates_evaluated"] == 1 << qubo["num_variables"]
assert qubo == qubo_snapshot
assert exact_config == exact_config_snapshot

print(json.dumps(exact_result, indent=2, ensure_ascii=False))

{
  "schema": "qubo-result.v1",
  "problem_id": "notebook-small-qubo",
  "solver": {
    "name": "exact-enumeration",
    "version": "0.1.0",
    "backend": "local-cpu"
  },
  "status": "optimal",
  "best_sample": [
    1,
    0,
    0
  ],
  "best_energy": -2,
  "runtime_seconds": 0.00027059996500611305,
  "metadata": {
    "algorithm": "exhaustive_enumeration"
  },
  "termination_reason": "search_exhausted",
  "metrics": {
    "candidates_evaluated": 8,
    "total_candidates": 8,
    "search_space_exhausted": true
  }
}


## 5. QAOA solver

这里使用 NumPy statevector 参考实现。固定 seed 保证实验可重复；`shots` 模式会在观测到的状态中返回最低能量样本。

In [5]:
qaoa_config = {
    "layers": 2,
    "optimizer_iterations": 20,
    "restarts": 4,
    "shots": 4096,
    "seed": 7,
    "max_variables": 10,
}

qaoa_config_snapshot = copy.deepcopy(qaoa_config)
qaoa_solver = QaoaQuboSolver()
qaoa_result = qaoa_solver.solve(qubo, config=qaoa_config)
qaoa_repeat = qaoa_solver.solve(qubo, config=qaoa_config)
validate_qubo_result(qubo, qaoa_result)

assert qaoa_result["status"] == "feasible"
assert qaoa_result["best_sample"] is not None
qaoa_energy_exact = evaluate_qubo_energy(
    qubo,
    qaoa_result["best_sample"],
)
assert qaoa_energy_exact >= oracle_energy
assert qaoa_result["best_energy"] == public_json_number(qaoa_energy_exact)
for field in ("best_sample", "best_energy", "metrics", "metadata"):
    assert qaoa_result[field] == qaoa_repeat[field]
assert qubo == qubo_snapshot
assert qaoa_config == qaoa_config_snapshot

print(json.dumps(qaoa_result, indent=2, ensure_ascii=False))

{
  "schema": "qubo-result.v1",
  "problem_id": "notebook-small-qubo",
  "solver": {
    "name": "qaoa-statevector",
    "version": "0.1.0",
    "backend": "numpy-statevector"
  },
  "status": "feasible",
  "best_sample": [
    1,
    0,
    0
  ],
  "best_energy": -2,
  "runtime_seconds": 0.07649680000031367,
  "metadata": {
    "algorithm": "qaoa",
    "layers": 2,
    "shots": 4096,
    "seed": 7,
    "parameters": [
      1.4210660439498732,
      2.3878010807566876,
      6.115962577471107,
      0.7596646003981559
    ]
  },
  "termination_reason": "optimizer_completed",
  "metrics": {
    "expectation": -1.3914678654784534,
    "optimizer_evaluations": 644,
    "selected_probability": 0.5417816798938587,
    "unique_samples_observed": 7
  }
}


## 6. 对比结果

Exact 必须与 oracle 完全一致。QAOA 的 gap 必须非负；它可能命中最优解，但依然只声明 `feasible`，因为采样本身不构成全局最优性证明。

In [6]:
comparison = [
    {
        "solver": "independent enumeration",
        "status": "proved optimal",
        "sample": oracle_sample,
        "energy": float(oracle_energy),
        "gap": 0.0,
    },
    {
        "solver": exact_result["solver"]["name"],
        "status": exact_result["status"],
        "sample": exact_result["best_sample"],
        "energy": exact_result["best_energy"],
        "gap": float(exact_energy_exact - oracle_energy),
    },
    {
        "solver": qaoa_result["solver"]["name"],
        "status": qaoa_result["status"],
        "sample": qaoa_result["best_sample"],
        "energy": qaoa_result["best_energy"],
        "gap": float(qaoa_energy_exact - oracle_energy),
    },
]

for row in comparison:
    print(row)

assert comparison[1]["gap"] == 0.0
assert comparison[2]["gap"] >= 0.0

{'solver': 'independent enumeration', 'status': 'proved optimal', 'sample': [1, 0, 0], 'energy': -2.0, 'gap': 0.0}
{'solver': 'exact-enumeration', 'status': 'optimal', 'sample': [1, 0, 0], 'energy': -2, 'gap': 0.0}
{'solver': 'qaoa-statevector', 'status': 'feasible', 'sample': [1, 0, 0], 'energy': -2, 'gap': 0.0}


## 7. 负例：输入和输出都不能绕过契约

先给 QUBO 添加重复稀疏项，再篡改 solver 返回的 energy。两者都必须在公共契约边界被拒绝。

In [7]:
invalid_qubo = copy.deepcopy(qubo)
invalid_qubo["terms"].append([0, 0, -3.0])

validation_error = None
try:
    validate_qubo(invalid_qubo)
except (TypeError, ValueError) as error:
    validation_error = str(error)

assert validation_error is not None
print("Expected input validation error:", validation_error)

tampered_result = copy.deepcopy(qaoa_result)
tampered_result["best_energy"] += 1
result_error = None
try:
    validate_qubo_result(qubo, tampered_result)
except (TypeError, ValueError) as error:
    result_error = str(error)

assert result_error is not None
print("Expected result validation error:", result_error)

Expected input validation error: QUBO terms contains duplicate index pairs.
Expected result validation error: best_energy does not match the energy recomputed from its sample.


## 结论

这套小规模校验确认了：

- `qubo.v1` 在运行前通过严格契约；
- Exact solver 与独立穷举 oracle 完全一致；
- QAOA 返回合法、可重算的 `qubo-result.v1`，但保守地保持 `feasible`；
- 坏的稀疏项会在 solver 边界之前被拒绝。

复用这本模板时，请替换 `qubo`，并同步更新 fixture 专属的预期 sample/energy 断言；候选数量等通用断言会从模型维度自动推导。